# 九州工業大学大学院生命体工学研究科 インターンシップ — 第2部 後半
## グラフ分類 — その精度は本当にGNNのおかげか

### 第2部後半の目的

ここまでは**1つの大きなグラフの中の頂点を分類する**問題を扱ってきた（頂点分類）。
第2部後半では問題設定を変え、**グラフ1つを丸ごと1個のデータとみなして分類する**（グラフ分類）。

題材は **PROTEINS** データセットである。
1113個のタンパク質をグラフとして表現したもので、
頂点=アミノ酸の二次構造要素、辺=それらが空間的に近いこと、
ラベル=そのタンパク質が酵素かどうか、を表す。

グラフ分類では、頂点ごとの特徴をグラフ1つ分にまとめる操作が必要になる。
これを **読み出し (readout)** または **プーリング** と呼び、ここでも集約のしかたが問題になる。

そして第2部後半の本題はもう一つある。

> **その精度は、本当にGNNがグラフ構造を理解したおかげなのか。**

GNNを組んで「精度75%が出ました」と報告する前に、
**それより単純な方法で何%出るのか**を必ず先に確かめなければならない。
これを**ベースライン**と呼ぶ。ベースラインを置かない性能報告には意味がない。

### 到達目標
1. グラフ分類における読み出し(プーリング)の役割を説明できる
2. 自分でベースラインを構築し、GNNと**同一の分割**で公正に比較できる
3. レイヤと読み出しの選択が、ベースラインを超えられるかどうかを左右することを示せる

In [ ]:
import os, time
import numpy as np
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (GCNConv, GraphConv, GINConv,
                                global_mean_pool, global_add_pool, global_max_pool)

def find_data_root():
    # 既定ではノートブックから見て ../data にデータを置く。
    # 手元で別の配置を試すときのために、候補を順に探す。
    # 環境変数 JIKKEN_DATA があればそれを最優先する。
    env = os.environ.get("JIKKEN_DATA")
    if env:
        return env
    for cand in ["../data", "../../data", "../../../data", "data"]:
        if (os.path.isdir(os.path.join(cand, "Planetoid"))
                or os.path.isdir(os.path.join(cand, "TUDataset"))):
            return cand
    return "../data"          # 見つからないときの既定値

DATA_ROOT = find_data_root()

# 本実習で扱うモデルは小さいため、計算スレッドを増やすとかえって遅くなる
# (並列化のオーバーヘッドが計算量を上回るため。実測で1.4〜1.5倍の差が出る)。
# 1スレッドに固定して実行時間を短縮する。
torch.set_num_threads(1)

dataset = TUDataset(root=f"{DATA_ROOT}/TUDataset", name="PROTEINS")
print(f"グラフ数   : {len(dataset)}")
print(f"頂点の特徴 : {dataset.num_features} 次元")
print(f"クラス数   : {dataset.num_classes}")
print()
g = dataset[0]
print("1個目のグラフ:", g)
print(f"  頂点数 {g.num_nodes}, 辺数 {g.num_edges}, ラベル {int(g.y)}")

---
## 1. データを観察する — ベースラインの手がかりを探す

モデルを組む前に、**データそのものを見る**。
特にグラフ分類では、「グラフの大きさ」だけでラベルが当たってしまうことがよくある。

> **課題2b-1** 次のセルを実行し、PROTEINSのクラス分布と、
> クラスごとの平均頂点数・平均辺数を記録せよ。

In [ ]:
y       = np.array([int(d.y) for d in dataset])
n_nodes = np.array([d.num_nodes for d in dataset], dtype=float)
n_edges = np.array([d.num_edges for d in dataset], dtype=float)

counts = np.bincount(y)
print(f"クラス分布: クラス0 = {counts[0]} 個, クラス1 = {counts[1]} 個")
print(f"全体の頂点数: 最小 {int(n_nodes.min())}, 最大 {int(n_nodes.max())}, "
      f"平均 {n_nodes.mean():.1f}")
print()
print("クラスごとのグラフの大きさ:")
for c in [0, 1]:
    print(f"  クラス{c}: 平均頂点数 {n_nodes[y==c].mean():6.1f}, "
          f"平均辺数 {n_edges[y==c].mean():6.1f}")
print()
print("-> クラスによってグラフの大きさが大きく違う。")
print("   つまり『頂点数を見るだけ』でもある程度は当たってしまうはずである。")

---
## 2. ベースラインを2つ作る

### ベースライン1: 多数派を答えるだけ

何も学習せず、常に「個数の多い方のクラス」と答える。
**どんなモデルもこれを超えられなければ価値がない**、という下限である。

### ベースライン2: グラフの大きさだけを見る決定木

**グラフの構造をいっさい見ず**、頂点数と辺数という2つの数だけから分類する決定木。
GNNが「グラフ構造を理解した」と主張するなら、**最低でもこれを超えなければならない**。

### 公正な比較のために — 分割を揃える

ベースラインとGNNを比べるときは、**まったく同じ訓練/テスト分割**を使わなければならない。
分割が違えば、差が手法の差なのか分割の差なのか分からなくなる。
そこで、seed から分割を作る関数を用意し、両者で共有する。

In [ ]:
TEST_RATIO = 0.2

def make_split(seed):
    # seed から訓練/テストの添字を作る。ベースラインとGNNで共通に使う
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(dataset), generator=g).numpy()
    n_test = int(len(dataset) * TEST_RATIO)
    return perm[n_test:], perm[:n_test]      # (train_idx, test_idx)

tr_idx, te_idx = make_split(0)
print(f"訓練 {len(tr_idx)} グラフ / テスト {len(te_idx)} グラフ")
print("テスト集合の最初の10個の添字:", te_idx[:10])

# --- 検算: 訓練とテストが重なっていないか、全グラフを使い切っているか ---
for _s in range(5):
    _tr, _te = make_split(_s)
    assert len(set(_tr) & set(_te)) == 0, "訓練集合とテスト集合が重複している"
    assert len(_tr) + len(_te) == len(dataset), "全グラフを使い切っていない"
    assert len(_te) == int(len(dataset) * TEST_RATIO), "テスト集合の大きさが違う"
print("分割の検算OK (重複なし、全グラフを使用、テスト比率も正しい)")

> **課題2b-2** 次のセルの空欄を埋めて、2つのベースラインを完成させよ。

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# グラフの構造を見ない特徴: 頂点数と辺数だけ
X_simple = np.c_[n_nodes, n_edges]

SEEDS = [0, 1, 2, 3, 4]
acc_major, acc_tree = [], []

for s in SEEDS:
    tr, te = make_split(s)

    # ----- ここから (a): 訓練集合の多数派クラスを答え続けたときのテスト精度 -----
    # ヒント: np.bincount(...).argmax() で最頻クラスが求まる
    major_class =
    acc_major.append(                    )
    # ----- ここまで -----

    # ----- ここから (b): 頂点数と辺数だけの決定木を訓練し、テスト精度を求める -----
    # ヒント: clf.fit(特徴, ラベル) で学習、clf.predict(特徴) で予測。
    #   学習に使ってよいのは訓練集合 tr だけである。
    #   テスト集合 te のラベルを学習に使ってはいけない(答えを見て答えることになる)。
    clf = DecisionTreeClassifier(max_depth=3, random_state=0)
    clf.fit(                    )
    pred_tree =
    acc_tree.append(                    )
    # ----- ここまで -----

acc_major, acc_tree = np.array(acc_major), np.array(acc_tree)
print(f"ベースライン1 多数派を答えるだけ      : "
      f"{acc_major.mean()*100:5.2f} % ± {acc_major.std()*100:4.2f}")
print(f"ベースライン2 頂点数と辺数だけの決定木: "
      f"{acc_tree.mean()*100:5.2f} % ± {acc_tree.std()*100:4.2f}")
print()
print("GNN はこの 2 つ目の数値を超えなければならない。")

In [ ]:
# --- 検算 ---
assert len(acc_major) == len(SEEDS) and len(acc_tree) == len(SEEDS), \
    "seed の数だけ結果が入っていない"
# 多数派ベースラインは、テスト集合における多数派クラスの割合と一致するはず
for _i, _s in enumerate(SEEDS):
    _tr, _te = make_split(_s)
    _mc = np.bincount(y[_tr]).argmax()
    assert np.isclose(acc_major[_i], (y[_te] == _mc).mean()), \
        "多数派ベースラインの計算が合わない"
# 決定木は多数派ベースラインを明確に上回るはず (大きさに情報があるため)。
assert acc_tree.mean() > acc_major.mean() + 0.05, \
    "決定木が多数派ベースラインをほとんど超えていない。" \
    "clf.fit と clf.predict に渡す添字を確認すること"

# --- ここからが重要な検算 ---
# テスト集合のラベルで学習してしまうと、精度は高く出るので気づきにくい。
# 決定木が「何件のデータで学習されたか」を直接確かめる。
_tr, _te = make_split(SEEDS[-1])
_n_fit = clf.tree_.n_node_samples[0]      # 木の根にあるサンプル数 = 学習に使った件数
assert _n_fit == len(_tr), (
    f"決定木が {_n_fit} 件で学習されている。訓練集合は {len(_tr)} 件のはずである。"
    " テスト集合で学習していないか、clf.fit に渡す添字を確認すること")

# 予測はテスト集合に対して行われていなければならない。
assert len(pred_tree) == len(_te), (
    f"予測が {len(pred_tree)} 件ある。テスト集合は {len(_te)} 件のはずである。"
    " clf.predict に渡す添字を確認すること")

# 精度が「予測とテスト集合の正解ラベルの一致率」になっているかを確かめる。
# うっかり y[te] どうしを比べると必ず 100% になるが、それは精度ではない。
assert np.isclose(acc_tree[-1], (pred_tree == y[_te]).mean()), (
    f"最後のseedの精度 {acc_tree[-1]:.4f} が、予測と正解の一致率と合わない。"
    " 何と何を比べているか確認すること")

# 頂点数と辺数の2つだけを見る深さ3の決定木が全問正解することはあり得ない。
assert acc_tree.max() < 0.95, (
    f"精度が {acc_tree.max()*100:.1f}% と高すぎる。"
    " 正解ラベルどうしを比べていないか確認すること")

print(f"ベースラインの検算OK (決定木は訓練{_n_fit}件で学習し、テスト{len(pred_tree)}件を予測)")

---
## 3. グラフ分類のGNN — 読み出し(プーリング)

頂点分類では、各頂点の特徴ベクトルをそのまま分類に使えた。
グラフ分類では**グラフ1個につき1本のベクトル**が必要である。
そこで畳み込みを終えたあと、グラフ内の全頂点の特徴を1本にまとめる。これが読み出しである。

| 関数 | 計算 | 性質 |
|---|---|---|
| `global_add_pool` | 全頂点の**和** | 頂点数に比例してスケールが変わる |
| `global_mean_pool` | 全頂点の**平均** | 頂点数によるスケールの違いが打ち消される |
| `global_max_pool` | 各次元の**最大** | 「際立った頂点があるか」を拾う |

第1節で、このデータはクラスによってグラフの大きさが違うことを確認した。
**どの読み出しが有利になるか、実験の前に予想を立てること。**

なお「平均を取ればグラフの大きさの情報が完全に消える」わけではない。
消えるのは**足し合わせによるスケールの違い**である。
畳み込みを通った後の各頂点の特徴には、
その頂点の周囲がどれだけ混み合っているかが反映されうるので、
大きさに関する手がかりが平均後も残ることはある。
「何が消えて何が残るのか」を区別して考えること。

### ミニバッチ処理について

グラフはそれぞれ大きさが違うので、画像のように単純に積み上げられない。
PyTorch Geometric は複数のグラフを**1つの大きな非連結グラフ**として連結し、
`batch` という配列で「各頂点がどのグラフに属するか」を記録する。
読み出し関数はこの `batch` を見てグラフごとにまとめている。

In [ ]:
loader_demo = DataLoader(dataset[:4], batch_size=4)
b = next(iter(loader_demo))
print("4つのグラフをまとめたバッチ:", b)
print(f"  合計頂点数: {b.num_nodes}")
print(f"  batch 配列の中身 (先頭30個): {b.batch[:30].tolist()}")
print(f"  各グラフの頂点数: {torch.bincount(b.batch).tolist()}")
print(f"  ラベル: {b.y.tolist()}")

In [ ]:
def make_conv(name, in_dim, out_dim):
    if name == "GCNConv":
        return GCNConv(in_dim, out_dim)
    if name == "GraphConv":
        return GraphConv(in_dim, out_dim)
    if name == "GINConv":
        return GINConv(Sequential(Linear(in_dim, out_dim), ReLU(),
                                  Linear(out_dim, out_dim)))
    raise ValueError(name)

POOLS = {"mean": global_mean_pool, "add": global_add_pool, "max": global_max_pool}


class GraphNet(torch.nn.Module):
    # 畳み込み3層 + 読み出し + 線形層
    def __init__(self, conv_name, pool_name, in_dim, hid, out_dim):
        super().__init__()
        self.conv1 = make_conv(conv_name, in_dim, hid)
        self.conv2 = make_conv(conv_name, hid, hid)
        self.conv3 = make_conv(conv_name, hid, hid)
        self.pool  = POOLS[pool_name]
        self.lin   = Linear(hid, out_dim)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = self.conv3(x, edge_index)
        x = self.pool(x, batch)                       # グラフ1個につき1本にまとめる
        x = F.dropout(x, p=0.5, training=self.training)
        return self.lin(x)


def train_and_test(conv_name, pool_name, seed, epochs=80, hid=64, bs=64):
    torch.manual_seed(seed)
    tr_idx, te_idx = make_split(seed)             # ベースラインと同じ分割
    tr_loader = DataLoader(dataset[tr_idx.tolist()], batch_size=bs, shuffle=True)
    te_loader = DataLoader(dataset[te_idx.tolist()], batch_size=bs)

    model = GraphNet(conv_name, pool_name, dataset.num_features,
                     hid, dataset.num_classes)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)

    for _ in range(epochs):
        model.train()
        for d in tr_loader:
            opt.zero_grad()
            loss = F.cross_entropy(model(d.x, d.edge_index, d.batch), d.y)
            loss.backward()
            opt.step()

    def accuracy(loader):
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for d in loader:
                pred = model(d.x, d.edge_index, d.batch).argmax(dim=1)
                correct += int((pred == d.y).sum()); total += d.num_graphs
        return correct / total

    return accuracy(tr_loader), accuracy(te_loader)

print("定義した")

---
## 4. まず1つ試す — 標準的なGCNで何%出るか

最も標準的な組み合わせ（`GCNConv` + 平均プーリング）を試す。

In [ ]:
t0 = time.time()
r = np.array([train_and_test("GCNConv", "mean", s) for s in SEEDS])
gcn_mean = r[:, 1]

print(f"GCNConv + mean プーリング")
print(f"  訓練精度  : {r[:,0].mean()*100:5.2f} %")
print(f"  テスト精度: {gcn_mean.mean()*100:5.2f} % ± {gcn_mean.std()*100:4.2f}")
print(f"  所要時間  : {time.time()-t0:.1f}s")
print()
print(f"  ベースライン2 (決定木): {acc_tree.mean()*100:5.2f} %")
diff = (gcn_mean.mean() - acc_tree.mean()) * 100
print(f"  差: {diff:+.2f} pt")
print()
if diff > 0:
    print("決定木を超えた。")
else:
    print("決定木に負けている。")
    print("グラフ構造を一切見ない2つの数値だけの決定木に、GNNが負けたことになる。")
    print("この結果を『GNNでグラフ分類ができた』と報告してはいけない。")

---
## 5. レイヤと読み出しを変える

第4節の結果を受けて、畳み込みレイヤと読み出しの組み合わせを網羅的に試す。

比較する3つのレイヤの違いは、**隣人の情報と自分自身の情報をどう扱うか**にある。

- **`GCNConv`**: 自分と隣人を区別せず、次数で正規化して平均的に混ぜる
  $$\bm{x}'_i = \sum_{j \in \mathcal{N}(i) \cup \{i\}} \frac{1}{\sqrt{\tilde d_i \tilde d_j}} W\bm{x}_j$$

- **`GraphConv`**: 自分と隣人に**別々の重み**を与え、正規化せずに足す
  $$\bm{x}'_i = W_1\bm{x}_i + W_2 \sum_{j \in \mathcal{N}(i)} \bm{x}_j$$

- **`GINConv`**: 隣人を**和**で集約したあとMLPを通す。
  理論上、グラフを区別する能力が3つの中で最も高いことが知られている
  $$\bm{x}'_i = \mathrm{MLP}\Big((1+\epsilon)\bm{x}_i + \sum_{j \in \mathcal{N}(i)} \bm{x}_j\Big)$$

3つの式の違いを、次の点に注目して読むこと。

- `GCNConv` は正規化によって**次数による大きさの違いを抑えている**。
  次数の情報が消えるわけではない（正規化係数そのものが次数から作られている）。
  抑えられるのは「隣人が多いほど値が大きくなる」というスケールの偏りである。
- `GraphConv` は正規化を行わず、自分と隣人に別々の重みを与える。
- `GINConv` は和で集約したあとMLPを通す。

**式の前提**: 上の3式は要点を見やすくするため簡略化してある。
実際の `GCNConv` と `GraphConv` は既定でバイアス項を持つ（式では省略した）。
`GINConv` の $\epsilon$ は既定で $0$ に固定されており、
本実習でも学習させずに $\epsilon = 0$ のまま用いる。

> **課題2b-3** 次のセルを実行し、9通りの組み合わせを比較せよ。数分かかる。

In [ ]:
CONVS = ["GCNConv", "GraphConv", "GINConv"]
POOL_NAMES = ["mean", "add", "max"]

# 第4節で測定済みの GCNConv+mean は再学習せず、その結果を使い回す
grid = {("GCNConv", "mean"): gcn_mean}

base = acc_tree.mean() * 100
print(f"決定木ベースライン = {base:.2f} %  (これを超えられるか)")
print()
print(f"{'レイヤ':<10} | " + " | ".join(f"{p+' pool':>16}" for p in POOL_NAMES))
print("-" * 68)

for cname in CONVS:
    row = f"{cname:<10} |"
    for pname in POOL_NAMES:
        if (cname, pname) in grid:
            te = grid[(cname, pname)]          # 第4節で測定済み
        else:
            r = np.array([train_and_test(cname, pname, s) for s in SEEDS])
            te = r[:, 1]
            grid[(cname, pname)] = te
        mark = "*" if te.mean() * 100 > base else " "
        row += f" {te.mean()*100:6.2f}% ±{te.std()*100:5.2f}{mark} |"
    print(row)

print()
print("* = 決定木ベースラインを上回った組み合わせ")

### 差の判定 — 同じ分割で比べていることを活かす

「平均が高い方が優れている」と言ってよいかを判定する。
ここで重要なのは、**ベースラインとGNNをまったく同じ分割で評価している**ことである。

分割が同じなら、seed ごとに1対1で差を取ることができる。これを**対応のある比較**と呼ぶ。
「たまたま簡単な分割を引いたから勝った」という可能性を排除できるので、
平均と標準偏差だけを比べるよりずっと信頼できる。

判定の手順は次のとおりである。

1. seed ごとに（GNNの精度）−（ベースラインの精度）を計算する
2. **5回すべてで正の差が出たか**を見る（全勝なら偶然とは考えにくい）
3. 差の平均を、差の**標準誤差** $\mathrm{sd}/\sqrt{n}$ と比べる

In [ ]:
def paired_compare(name_a, acc_a, name_b, acc_b):
    # 同一分割で得た2つの精度列を、seed ごとに対応させて比較する
    d = (np.array(acc_a) - np.array(acc_b)) * 100      # seed ごとの差 (pt)
    n = len(d)
    se = d.std(ddof=1) / np.sqrt(n)                    # 差の標準誤差

    print(f"{name_a}  vs  {name_b}")
    print(f"  {'seed':>6} | {name_a:>10} | {name_b:>10} | {'差':>8}")
    print("  " + "-" * 44)
    for i, s_ in enumerate(SEEDS):
        print(f"  {s_:>6} | {acc_a[i]*100:9.2f}% | {acc_b[i]*100:9.2f}% | {d[i]:+7.2f}pt")
    wins = int((d > 0).sum())
    print(f"  {'平均':>6} | {np.mean(acc_a)*100:9.2f}% | {np.mean(acc_b)*100:9.2f}% | "
          f"{d.mean():+7.2f}pt")
    print()
    print(f"  {name_a} が勝った回数: {wins} / {n}")
    print(f"  差の平均 {d.mean():+.2f} pt,  差の標準誤差 {se:.2f} pt")
    if wins == n and d.mean() > 2 * se:
        print("  判定: この5分割では一貫して高い (全勝かつ平均差が標準誤差の2倍超)")
    elif d.mean() > 2 * se:
        print("  判定: 高い傾向はあるが全勝ではない")
    else:
        print("  判定: 差があるとは言えない")
    print()


best = max(grid, key=lambda k: grid[k].mean())
print(f"最良の組み合わせ: {best[0]} + {best[1]} プーリング")
print(f"  テスト精度: {grid[best].mean()*100:.2f} % ± {grid[best].std()*100:.2f}")
print()

paired_compare(f"{best[0]}+{best[1]}", grid[best], "決定木", acc_tree)
paired_compare("GCNConv+mean", grid[("GCNConv", "mean")], "決定木", acc_tree)

worst = min(grid, key=lambda k: grid[k].mean())
print(f"最も悪かった組み合わせ: {worst[0]} + {worst[1]} "
      f"({grid[worst].mean()*100:.2f} % ± {grid[worst].std()*100:.2f})")
print()
paired_compare(f"{best[0]}+{best[1]}", grid[best],
               f"{worst[0]}+{worst[1]}", grid[worst])

---
## 6. 選んだ最良の組み合わせを、別のseedで確かめる

第5節では、9通りの中から最も平均精度の高い組み合わせを選び、
**その選ぶのに使ったのと同じ5つの分割**で決定木と比較して判定した。

**ここに落とし穴がある。**

9通りを試して最も良かったものを選んだのだから、その組み合わせは
**たまたまその5つの分割に合っていただけ**かもしれない。
候補をたくさん試せば、偶然よく当たるものが1つくらいは出てくる。
選ぶのに使った分割で、選んだものを評価すれば、成績は良く出て当たり前である。

これを避ける標準的な方法は、**選ぶための分割と、確かめるための分割を分ける**ことである。

- seed 0〜4 … 9通りから最良を**選ぶ**ために使った（第5節）
- seed 5〜9 … 選んだ1つを**確かめる**ために使う（この節。ここで初めて使う）

> **課題2b-4** 次のセルを実行し、第5節で選んだ組み合わせが、
> 選択に使っていないseedでも決定木を上回るかを確かめよ。約1分かかる。

In [ ]:
SEEDS_CONFIRM = [5, 6, 7, 8, 9]

# 確認用の分割で決定木ベースラインを測り直す
acc_tree_c = []
for s in SEEDS_CONFIRM:
    tr, te = make_split(s)
    clf = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_simple[tr], y[tr])
    acc_tree_c.append((clf.predict(X_simple[te]) == y[te]).mean())
acc_tree_c = np.array(acc_tree_c)

# 第5節で選んだ組み合わせ「だけ」を、確認用のseedで学習する
best = max(grid, key=lambda k: grid[k].mean())
acc_best_c = np.array([train_and_test(best[0], best[1], s)[1] for s in SEEDS_CONFIRM])

d = (acc_best_c - acc_tree_c) * 100
print(f"選んだ組み合わせ: {best[0]} + {best[1]}")
print()
print(f"  {'seed':>6} | {'GNN':>9} | {'決定木':>9} | {'差':>9}")
print("  " + "-" * 42)
for i, s in enumerate(SEEDS_CONFIRM):
    print(f"  {s:>6} | {acc_best_c[i]*100:8.2f}% | {acc_tree_c[i]*100:8.2f}% | {d[i]:+8.2f}pt")
print(f"  {'平均':>6} | {acc_best_c.mean()*100:8.2f}% | {acc_tree_c.mean()*100:8.2f}% | "
      f"{d.mean():+8.2f}pt")
print()
se = d.std(ddof=1) / np.sqrt(len(d))
print(f"  勝った回数: {int((d > 0).sum())} / {len(d)}")
print(f"  差の平均 {d.mean():+.2f} pt,  差の標準誤差 {se:.2f} pt")
print()
print(f"  選択に使った seed 0-4 での差: {(grid[best] - acc_tree).mean()*100:+.2f} pt")
print(f"  確認に使った seed 5-9 での差: {d.mean():+.2f} pt")

**読み取りかた**

確認用のseedでも決定木を上回ったなら、第5節の結論は補強される。
上回らなかった、あるいは差が小さくなったなら、
第5節の結果には**選び方による下駄が履かされていた**ということである。

どちらの結果になっても、そのまま正直に報告すること。
「選ぶのに使った分割で良い成績が出た」ことと「一般に優れている」ことは別である。

なお、これでもまだ十分ではない。
確認用の5分割も同じ PROTEINS から作ったもので、訓練集合どうしは大きく重なっている。
本当に一般的な優位性を主張するには、別のデータセットでの検証や、
比較対象と仮説を**測定する前に**決めておくことが必要になる。

---
## 7. 第2部後半の課題

### 実習課題（ノートブックで提出）

- **課題2b-1** PROTEINSのクラス分布とグラフの大きさの観察（第1節）
- **課題2b-2** 2つのベースラインの実装（第2節）
- **課題2b-3** 9通りの組み合わせの比較（第5節）
- **課題2b-4** 選択に使っていないseedでの確認（第6節）
- **課題2b-5** 最良の組み合わせについて、畳み込みの層数を 2, 3, 4, 5 と変えて測定せよ。
  第2部前半で頂点分類について観察したことと同じ傾向が見られるか。

In [ ]:
# 課題2b-5 をここに書く

---
## 8. 追加課題

ここまでの課題は、**測定して記録する**ことが中心だった。
以下の2問は、**測定した結果から何がどこまで言えるのか**を問うものである。
手を動かす必要はない。これまでに得た数値を根拠に、自分の言葉で述べよ。

### 追加課題2b-6（記述）

第6節で、選択に使った seed 0〜4 での差と、
選択に使っていない seed 5〜9 での差を比較した。

**(1)** 2つの差はどう違ったか。違いが小さかった場合、
それは「選択の影響がなかった」ことの証明になるか。

（ここに記述）

**(2)** 「この組み合わせは決定木より優れている」と主張するには、
ここまでの結果に加えてあと何が必要か。
第6節の末尾に挙げた注意点を踏まえて述べよ。

（ここに記述）

### 追加課題2b-7（記述）

本実習では、第1部後半以降、すべての精度を5回以上の平均と標準偏差で報告し、
対応のある比較で判定してきた。

**(1)** もし1回だけの実行で比較していたら、
どの結論が変わっていた可能性があるか。
自分の測定結果から具体例を1つ挙げて述べよ。

（ここに記述）

**(2)** 自分の研究で手法を比較するとき、
今日の進め方のうちどれを持ち帰れるか。1つ挙げて理由を述べよ。

（ここに記述）

### 参考文献

1. 村田 剛志. グラフニューラルネットワーク. オーム社, 2022.
2. 佐藤 竜馬. グラフニューラルネットワーク. 講談社, 2024.
3. 村田 剛志. Pythonで学ぶネットワーク分析. オーム社, 2019.
4. PyTorch Geometric ドキュメント. https://pytorch-geometric.readthedocs.io/